# TIL 2026-11-19 — Chunking과 검색 품질 (Lesson 07), pgvector 설치 (Lesson 06 이어서)

## 오늘 한 일
- 어제 HOLD였던 pgvector를 해결했다. Visual Studio Build Tools 2022(MSVC 14.44, `nmake`)가 이미 있어서 강의의 경로 B대로 v0.8.7을 빌드하고, 관리자 권한(UAC)으로 `nmake install`을 실행했다.
- `ai_agent_lesson06` DB에서 `CREATE EXTENSION vector`를 실행했고 `vector 0.8.7`이 활성화됐다.
- Lesson 07(Fixed / Paragraph / Overlap 청킹)의 STEP 07~13과 선택 실습 B, C, E, F를 실행했다.
- Lesson 06의 Fixture 단계(저장, 재조회, 중복 INSERT)를 실행했다. Live 단계는 실행하지 않았다.

## pgvector 설치 확인
| 항목 | 결과 |
|---|---|
| 빌드 (`nmake /F Makefile.win`) | PASS (`vector.dll` 생성) |
| 설치 (`nmake install`, 관리자 권한) | PASS (확장 파일 복사됨) |
| `CREATE EXTENSION vector` | PASS, 버전 0.8.7 |
| `vector(3)` 거리 테스트 | PASS (같은 방향 0.0, 직각 1.0, 테스트 테이블은 롤백) |

## Lesson 07 결과
| 항목 | 결과 |
|---|---|
| STEP 07~08 Fixed 160 | PASS: 4개 저장, 새 프로세스에서 검색됨 (1위 chunk 2, 거리 0.2929) |
| STEP 09~10 Paragraph / Overlap 30 | PASS: 5개 / 5개 저장 |
| 선택 B Overlap 60 | PASS: 7개 저장 |
| STEP 13 자동 테스트 | PASS: 16개 OK (오프라인 9 + DB 7, skip 없음) |
| 선택 C Live Paragraph | PASS: 5개 저장, 임베딩 요청 2번 |
| 선택 E 임베딩 비교 (OpenAI vs Gemini) | 실행함, 요청 4번. Hit@3 수치는 이 기록에 옮기지 못함 |
| 선택 D (Claude Code 설명 프롬프트) | NOT_RUN |

### STEP 11 비교표 (질문 "과제 마감일", 정답 문구 "금요일 오후 6시")
| 설정 | 저장 수 | Hit@3 | 정답 첫 순위 | Hit@1 |
|---|---|---|---|---|
| fixed:size=160:overlap=0 | 4 | 1 | 2위 | 0 |
| paragraph | 5 | 1 | 2위 | 0 |
| overlap 30 | 5 | 1 | 2위 | 0 |
| overlap 60 | 7 | 1 | **3위** | 0 |

- Hit@3은 네 설정 모두 1이지만, Hit@1은 모두 0이다. "Top-3 안에 들어왔다"와 "1위로 나왔다"는 다른 얘기다.
- Overlap을 60으로 키우면 청크가 7개로 늘고 정답 순위는 오히려 3위로 밀렸다. 겹침이 많다고 검색이 좋아지는 게 아니다.
- Fixed와 Paragraph는 0벡터 청크가 제외돼 결과가 3건이 아니라 2건이었다.

### Live 결과 (선택 C, `text-embedding-3-small`)
| 순위 | 청크 | 거리 |
|---|---|---|
| 1 | 질문 및 문의 | 0.6998 |
| 2 | 과제 제출 (정답 문구 포함) | 0.6999 |
| 3 | 평가와 근거 | 0.7041 |

- 1위와 2위 거리 차이가 0.0001이고, 세 거리가 모두 약 0.70이라 거의 분리되지 않는다.
- 정답 청크가 2위인 건 Fixture와 같다. 이 숫자만으로 Live가 더 낫다고 말할 수 없다.

## Lesson 06 STEP 12 기록표 (갱신)
| 항목 | 상태 | 근거 |
|---|---|---|
| pgvector 확장 | **PASS** | 0.8.7 활성화 |
| Fixture 저장·재조회 | PASS | `count` 3건. 마감일 / API Key 관리 / 문의 게시판 질문이 각각 대응 문단을 거리 0.0000으로 1위, 나머지 1.0000 |
| 중복 INSERT (Upsert) | PASS | `index`를 두 번 실행해도 3건 유지 |
| Live 선택 실행 | NOT_RUN | `live-index`, `live-query` 실행 안 함 (API 요청 약 3번 발생 예정) |
| 근거 검토 | NOT_RUN | Live를 안 해서 환불 질문 검토를 못 함 |

> 어제 TIL(11-18)의 pgvector HOLD 항목은 이 표로 대체된다.

## 겪은 문제
1. **내 확인 스크립트 실수**: 처음에 `fixture_vector`에 `(start, end, content)` 튜플을 통째로 넣어서 전부 0벡터가 나왔다. 코드가 아니라 호출이 틀렸다. `split_chunks`가 튜플 리스트를 돌려준다는 걸 확인하고 content만 넘겨서 해결했다.
2. **확장 생성이 롤백됨**: 스모크 테스트에서 `rollback()`이 같은 트랜잭션의 `CREATE EXTENSION`까지 되돌렸다. 확장은 먼저 커밋하고, 테스트 테이블은 별도 트랜잭션으로 분리했다.
3. **`python -I`는 `PYTHONUTF8`를 무시해서** 콘솔 한글이 깨졌다.
4. **PATH 누락**: `psycopg`를 쓰려면 매번 `C:\Program Files\PostgreSQL\18\bin`을 PATH에 넣어야 한다.
5. **DB 비밀번호 인증 실패**: 추측하지 않고 `.env`를 직접 고쳐 달라고 했다.

## 아직 안 끝난 것
- `document_qa.py`가 여전히 문법 오류 상태다. 전체 `unittest discover`는 3개 모듈(`test_document_qa`, `test_embedding_search`, `test_vector_store`)이 import 단계에서 실패한다. 오늘 Lesson 07 테스트 16개는 이 모듈들과 별개로 통과했다.
- `tests/embedding_search.py`가 `embedding_search.py`와 다른 내용으로 남아 있다.
- 아무것도 커밋하지 않았다.
- `%TEMP%\pgvector_lesson06`에 빌드용 소스 폴더가 남아 있다.

## 배운 것
- **청킹은 검색 품질을 바꾸지만 언제나 좋아지지는 않는다.** 같은 질문에 설정만 바꿔도 정답 순위가 2위에서 3위로 바뀌었다.
- **Hit@K는 문자열 포함 여부로 센 값**이다. 답변이 맞았다는 뜻이 아니다. Hit@3=1이어도 Hit@1=0이면 1위 근거로 바로 답하기엔 위험하다.
- **코사인 거리가 서로 비슷하면** 순위는 있어도 구분력이 없다. 순위만 보지 말고 거리 차이도 같이 봐야 한다.
- **0벡터는 검색에서 제외**해야 한다(`vector_norm > 0`). 코사인 거리가 정의되지 않는다.
- **설치와 활성화는 별개**다. 파일을 설치(`nmake install`)한 뒤에도 DB마다 `CREATE EXTENSION vector`가 필요하다.
- **트랜잭션 경계**를 의식해야 한다. 같은 트랜잭션에서 `rollback()`하면 앞선 DDL도 같이 사라진다.

## 확인 문제 — 내 답 (직접 채우기)
1. Hit@3=1인데 Hit@1=0이면 무슨 뜻인가? →
2. Overlap을 늘렸는데 정답 순위가 떨어질 수 있는 이유는? →
3. 거리가 0.6998, 0.6999, 0.7041이면 1위를 믿어도 되는가? →

## 다음에 할 일
- [ ] Lesson 06 Live 단계 (`live-index` → `live-query` → 환불 질문 근거 검토), 실행 전 API 호출 횟수 확인
- [ ] 선택 E의 Hit@3 수치를 다시 확인해서 기록
- [ ] `document_qa.py` 복구 결정 (형식만 복구 / `git restore`) 후 전체 테스트 재실행
- [ ] `tests/embedding_search.py` 정리
- [ ] 쌓인 변경과 TIL 파일 정리 후 커밋
- [ ] 선택 D (Claude Code 설명 프롬프트)

## 오늘의 한 줄
Hit@3=1만 보고 끝내지 않고 Hit@1과 거리 차이까지 같이 적었다. 검색이 됐다는 것과 근거로 쓸 만하다는 것은 다르다.
